In [1]:
pip install streamlit openai tensorflow scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import streamlit as st
import numpy as np
import pandas as pd
import openai
from openai import OpenAI
import os

In [3]:
@st.cache_resource
def load_model_and_tools():
    import joblib
    from tensorflow.keras.models import load_model
    scaler = joblib.load('scaler.joblib')
    le = joblib.load('label_encoder.joblib')
    model = load_model('best_model.keras')
    return scaler, le, model

scaler, le, model = load_model_and_tools()

openai.api_key = st.secrets["OPENAI_API_KEY"]

st.title("사용자별 수면 상태 예측 & 맞춤형 건강 코치")

uploaded_file = st.file_uploader("전체 사용자 건강 데이터 CSV 파일 업로드", type=['csv'])

if uploaded_file is not None:
    user_df_all = pd.read_csv(uploaded_file)
    st.write("데이터 샘플", user_df_all.head())

    # 사용자 ID 입력 받기 (예: 'user_id' 컬럼 기준)
    user_id = st.text_input("사용자 ID를 입력하세요")

    if user_id:
        # user_id 기준 데이터 필터링
        user_df = user_df_all[user_df_all['user_id'] == user_id]
        
        if len(user_df) < 14:
            st.error("해당 사용자의 데이터가 14일 이상 존재하지 않습니다.")
        else:
            user_recent = user_df.tail(14).copy()

            # 전처리
            features = ['TotalSteps', 'VeryActiveMinutes', 'FairlyActiveMinutes',
                        'LightlyActiveMinutes', 'SedentaryMinutes', 'VeryActiveDistance',
                        'ModeratelyActiveDistance', 'LightActiveDistance', 'Calories',
                        'TotalMinutesAsleep', 'TotalTimeInBed']

            try:
                user_recent['DayType_encoded'] = le.transform(user_recent['DayType'])
                X_num = scaler.transform(user_recent[features].values)
                X_combined = np.concatenate([X_num, user_recent['DayType_encoded'].values.reshape(-1,1)], axis=1)
                X_input = np.expand_dims(X_combined, axis=0)

                predictions = model.predict(X_input)
                predicted_labels = (predictions > 0.5).astype(int).flatten()

                st.subheader(f"사용자 {user_id} 향후 3일 간 예측된 수면 상태")
                st.write(predicted_labels.tolist(), " (0=불량, 1=양질)")

                user_summary_stats = {
                    '평균 수면시간': round(user_recent['TotalMinutesAsleep'].mean() / 60, 2),
                    '평균 수면 효율': round((user_recent['TotalMinutesAsleep'].sum() / user_recent['TotalTimeInBed'].sum()) * 100, 1),
                    '평균 걸음 수': int(user_recent['TotalSteps'].mean()),
                    '아주 활동적인 시간': round(user_recent['VeryActiveMinutes'].mean(), 1),
                    '앉아있는 시간': round(user_recent['SedentaryMinutes'].mean(), 1)
                }

                summary_input_text = f"""
                최근 14일 동안의 사용자 건강 데이터 요약:
                - 평균 수면시간: {user_summary_stats['평균 수면시간']}시간
                - 평균 수면 효율: {user_summary_stats['평균 수면 효율']}%
                - 평균 걸음 수: {user_summary_stats['평균 걸음 수']}보
                - 아주 활동적인 시간: {user_summary_stats['아주 활동적인 시간']}분
                - 앉아있는 시간: 하루 평균 {user_summary_stats['앉아있는 시간']}분
                """

                st.subheader("사용자 건강 데이터 요약")
                st.text(summary_input_text)

                full_prompt = f"""
                안녕하세요, 수면 건강 코치입니다.

                {summary_input_text}

                향후 3일 간 예측된 수면 상태 (0=불량, 1=양질): {predicted_labels.tolist()}

                이 데이터를 기반으로 친절하고 전문적인 피드백과 행동 추천, 동기부여 메시지를 만들어 주세요.
                """

                if st.button("맞춤형 건강 피드백 받기"):
                    with st.spinner("OpenAI API 호출 중..."):
                        response = openai.ChatCompletion.create(
                            model="gpt-4o-mini",
                            messages=[
                                {"role": "system", "content": "당신은 친절한 수면 건강 코치입니다."},
                                {"role": "user", "content": full_prompt},
                            ],
                            temperature=0.7,
                            max_tokens=500,
                        )
                        advice = response.choices[0].message.content.strip()
                        st.subheader("건강 코치 피드백")
                        st.write(advice)

            except Exception as e:
                st.error(f"전처리 또는 예측 중 오류: {e}")

else:
    st.info("전체 사용자 데이터 CSV 파일을 업로드해주세요.")

2025-07-15 23:18:23.833 
  command:

    streamlit run C:\Users\choin\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]


FileNotFoundError: [Errno 2] No such file or directory: 'scaler.joblib'